# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sanaullah-Turab/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth a refresh review if it still gets real search
traffic, hasn't been touched in at least three months, and its click-through rate is worse
than what pages sitting in the same position bracket normally earn. Low CTR at position 40 is
normal. The same CTR at position 5 is a problem — the flag has to know the difference.

**Reason code:** `stale_ctr_underperformer` — one reason code, because this baseline encodes
one rule, not the six-flag combo the FlyRank session showed. **Action label:**
`refresh_review` when the rule fires, `monitor` otherwise.

Before coding it, two signals get checked, because the session's lesson was that a rule is
only as honest as the signals under it:

1. **Staleness → decline.** Behind FlyRank's `stale_visible_page` refresh flag: does
   `days_since_last_update` actually track with `trend_direction == "down"`?
2. **CTR vs. position.** Behind FlyRank's `low_ctr_visible_page` / CTR-fix logic: does CTR
   really fall as position gets worse, and by how much per tier — so a flat 0.5% threshold can
   be replaced with a tier-aware one?


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.width", 120)

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"rows: {len(df):,} | columns: {df.shape[1]}")

# ---------------------------------------------------------------
# Signal 1 -- staleness vs. decline rate (behind stale_visible_page / refresh flags)
# Claim: "the longer a page has gone unupdated, the more likely it's declining."
# ---------------------------------------------------------------
signal1 = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"),
           down_rate=("trend_direction", lambda s: (s == "down").mean()))
      .reindex(["0-30", "31-90", "91-180", "181+"])
)
signal1["down_rate"] = signal1["down_rate"].round(3)
print("\nSIGNAL 1 -- staleness vs. decline rate, by freshness_tier (n printed)")
print(signal1)

# floor check: every bucket needs >= 50 rows before a verdict is allowed
assert (signal1["n"] >= 50).all(), "a freshness_tier bucket is under the 50-row floor"

verdict_1 = "MIXED"
print(f"\nVERDICT (staleness -> decline): {verdict_1}")
print("down_rate rises from 0-30 (51.1%) through 91-180 (61.1%), which is the story the")
print("refresh flag assumes -- but the most-stale bucket, 181+, drops back to 47.1% (n=174,")
print("above the floor, not noise). Staleness alone does not cleanly predict decline, so it")
print("cannot be the sole driver of the rule below -- it earns a place as a gate, not a score.")

# ---------------------------------------------------------------
# Signal 2 -- CTR vs. position (behind low_ctr_visible_page / CTR-fix logic)
# Claim: "pages in better position tiers get higher CTR."
# Weighted CTR (sum clicks / sum impressions), not a mean of per-page ratios -- a raw mean
# is dominated by page one-off rows with near-zero impressions (see the n and note below).
# ---------------------------------------------------------------
visible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)]
signal2 = (
    visible.groupby("position_tier")
           .agg(n=("content_id", "size"),
                total_clicks=("clicks_90d", "sum"),
                total_impressions=("impressions_90d", "sum"))
           .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
)
signal2["weighted_ctr_pct"] = (signal2["total_clicks"] / signal2["total_impressions"] * 100).round(3)
print("\nSIGNAL 2 -- weighted CTR vs. position_tier, visible pages only (n printed)")
print(signal2)

assert (signal2["n"] >= 50).all(), "a position_tier bucket is under the 50-row floor"

verdict_2 = "CONFIRMED"
print(f"\nVERDICT (CTR vs. position): {verdict_2}")
print("weighted CTR falls in step with position, top_3 (0.49%) down to deep (0.04%) --")
print("the CTR-fix logic's assumption holds. That means a single flat CTR threshold (e.g.")
print("'ctr < 0.5') mislabels every low-position page as broken when low CTR there is normal.")
print("The rule below compares each page's CTR against its OWN tier's weighted baseline")
print("instead of one global cutoff.")

tier_baseline_ctr = signal2["weighted_ctr_pct"].to_dict()
tier_baseline_ctr


rows: 30,000 | columns: 44

SIGNAL 1 -- staleness vs. decline rate, by freshness_tier (n printed)
                    n  down_rate
freshness_tier                  
0-30            20480      0.511
31-90             175      0.589
91-180           9171      0.611
181+              174      0.471

VERDICT (staleness -> decline): MIXED
down_rate rises from 0-30 (51.1%) through 91-180 (61.1%), which is the story the
refresh flag assumes -- but the most-stale bucket, 181+, drops back to 47.1% (n=174,
above the floor, not noise). Staleness alone does not cleanly predict decline, so it
cannot be the sole driver of the rule below -- it earns a place as a gate, not a score.

SIGNAL 2 -- weighted CTR vs. position_tier, visible pages only (n printed)
                  n  total_clicks  total_impressions  weighted_ctr_pct
position_tier                                                         
top_3           533         34222            7025180             0.487
page_1         8633        313097    

{'top_3': 0.487,
 'page_1': 0.35,
 'striking': 0.347,
 'page_3_5': 0.155,
 'deep': 0.039}

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

Gate: `visible` (real traffic) AND `stale_enough` (untouched 90+ days -- Signal 1 said "gate,
not driver", so this is a floor, not the 180-day extreme that MIXED result) AND
`underperforms_ctr` (CTR below its own position tier's weighted baseline from Signal 2).
Score is `impressions_90d` on triggered rows -- readable on purpose: it ranks by how much
traffic is riding on the underperformance. No fitted weights.


In [2]:
import os

has_position = df["avg_position"] > 0
visible_mask = df["impressions_90d"] >= 500
stale_enough = df["days_since_last_update"] >= 90
expected_ctr_for_tier = df["position_tier"].map(tier_baseline_ctr)
underperforms_ctr = has_position & (df["ctr"] < expected_ctr_for_tier)

trigger = visible_mask & stale_enough & underperforms_ctr
print(f"triggered rows: {trigger.sum():,} / {len(df):,} ({trigger.mean():.1%})")

queue_cols = ["content_id", "client_id", "content_type", "impressions_90d", "clicks_90d",
              "ctr", "avg_position", "position_tier", "days_since_last_update",
              "freshness_tier", "trend_direction", "word_count"]

queue = df[queue_cols].copy()
queue["expected_ctr_for_tier"] = expected_ctr_for_tier.round(3)
queue["score"] = np.where(trigger, df["impressions_90d"], 0)
queue["reason_code"] = np.where(trigger, "stale_ctr_underperformer", "no_flag")
queue["action"] = np.where(trigger, "refresh_review", "monitor")

queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
out_path = "../outputs/baseline_action_score.csv"
queue.to_csv(out_path, index=False)
print(f"wrote {len(queue):,} rows to {out_path}")

queue.head(10)


triggered rows: 4,560 / 30,000 (15.2%)


wrote 30,000 rows to ../outputs/baseline_action_score.csv


,content_id,client_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,position_tier,days_since_last_update,freshness_tier,trend_direction,word_count,expected_ctr_for_tier,score,reason_code,action
0,content_5fe46e04994d,client_4e07408562,keyword article,517715,741,0.14,4.2,page_1,104,91-180,down,NaN,0.350,517715,stale_ctr_underperformer,refresh_review
1,content_cb112fce36be,client_19581e27de,keyword article,309910,492,0.16,5.6,page_1,104,91-180,down,2761.0,0.350,309910,stale_ctr_underperformer,refresh_review
2,content_36ff89c8214e,client_19581e27de,keyword article,295097,154,0.05,7.3,page_1,104,91-180,stable,NaN,0.350,295097,stale_ctr_underperformer,refresh_review
3,content_b28d1efd668f,client_6208ef0f77,keyword article,286608,169,0.06,26.2,page_3_5,104,91-180,stable,6901.0,0.155,286608,stale_ctr_underperformer,refresh_review
4,content_813e88069237,client_6208ef0f77,keyword article,233561,129,0.06,26.2,page_3_5,104,91-180,down,4610.0,0.155,233561,stale_ctr_underperformer,refresh_review
5,content_c8e9d6ab9013,client_19581e27de,keyword article,208678,0,0.00,9.7,page_1,104,91-180,down,NaN,0.350,208678,stale_ctr_underperformer,refresh_review
6,content_b511d4bc4ad2,client_6208ef0f77,keyword article,205915,290,0.14,27.9,page_3_5,104,91-180,stable,5591.0,0.155,205915,stale_ctr_underperformer,refresh_review
7,content_d17681677e69,client_19581e27de,keyword article,201584,487,0.24,5.8,page_1,104,91-180,stable,NaN,0.350,201584,stale_ctr_underperformer,refresh_review
8,content_a7427266c305,client_19581e27de,keyword article,201111,219,0.11,5.7,page_1,104,91-180,stable,NaN,0.350,201111,stale_ctr_underperformer,refresh_review
9,content_c5063073d048,client_6208ef0f77,keyword article,192205,466,0.24,12.5,striking,104,91-180,stable,5127.0,0.347,192205,stale_ctr_underperformer,refresh_review


## 3. Top-10 review

*For each of the top ten: the action, why it's there, and what would make it wrong.*


In [3]:
top10 = queue.head(10).reset_index(drop=True)
top10

# One line each, written by hand after reading the row above: action / why / what would
# make it wrong.
top10_review = [
    "refresh_review -- 517k impressions, page_1 (4.2) but CTR 0.14% vs 0.35% tier baseline; "
    "wrong if the drop is a SERP-feature eating clicks (featured snippet/AI overview) that a "
    "content refresh can't fix.",

    "refresh_review -- 310k impressions, page_1 (5.6), CTR 0.16% vs 0.35%, already trending "
    "down; wrong if trend_direction=='down' here is seasonal and reverses on its own next window.",

    "refresh_review -- 295k impressions, page_1 (7.3), CTR 0.05% is far below 0.35%; wrong if "
    "the title/meta shown in search no longer matches the page (a title-tag fix, not a refresh).",

    "refresh_review -- 287k impressions, page_3_5 (26.2), CTR 0.06% vs 0.15% tier baseline; "
    "wrong if this page is a poor keyword match for the query mix driving those impressions, "
    "so no amount of refresh moves the CTR.",

    "refresh_review -- 234k impressions, page_3_5 (26.2), CTR 0.06%, trend already down; wrong "
    "if the page is being cannibalized by a newer sibling page ranking for the same query.",

    "refresh_review -- 209k impressions, page_1 (9.7), CTR 0.00% (zero of 208k clicks); wrong "
    "if this is a tracking gap (GSC/GA4 mismatch) rather than a real zero-click page -- worth a "
    "sanity check against clicks_last_30d before assuming it's real.",

    "refresh_review -- 206k impressions, page_3_5 (27.9), CTR 0.14% vs 0.15% -- barely under "
    "the tier baseline; wrong because this gap is small enough to be noise, not a real "
    "underperformer -- weakest pick in the top 10.",

    "refresh_review -- 202k impressions, page_1 (5.8), CTR 0.24% vs 0.35%; wrong if the query "
    "mix behind these impressions is mostly navigational (branded searches with naturally low "
    "CTR), which the position tier alone can't detect.",

    "refresh_review -- 201k impressions, page_1 (5.7), CTR 0.11% vs 0.35%; wrong if word_count "
    "is missing/blank for this row (content_type-driven missingness) and thinness, not staleness, "
    "is the real driver -- worth checking has_word_count before acting.",

    "refresh_review -- 192k impressions, striking (12.5), CTR 0.24% vs 0.35%; wrong if this "
    "page recently moved from page_1 into striking distance and the CTR gap is a temporary "
    "position-shift artifact rather than a stable underperformance.",
]

for i, (row, note) in enumerate(zip(top10.itertuples(), top10_review), start=1):
    print(f"{i}. [{row.content_id}] score={row.score:,} | {note}")


1. [content_5fe46e04994d] score=517,715 | refresh_review -- 517k impressions, page_1 (4.2) but CTR 0.14% vs 0.35% tier baseline; wrong if the drop is a SERP-feature eating clicks (featured snippet/AI overview) that a content refresh can't fix.
2. [content_cb112fce36be] score=309,910 | refresh_review -- 310k impressions, page_1 (5.6), CTR 0.16% vs 0.35%, already trending down; wrong if trend_direction=='down' here is seasonal and reverses on its own next window.
3. [content_36ff89c8214e] score=295,097 | refresh_review -- 295k impressions, page_1 (7.3), CTR 0.05% is far below 0.35%; wrong if the title/meta shown in search no longer matches the page (a title-tag fix, not a refresh).
4. [content_b28d1efd668f] score=286,608 | refresh_review -- 287k impressions, page_3_5 (26.2), CTR 0.06% vs 0.15% tier baseline; wrong if this page is a poor keyword match for the query mix driving those impressions, so no amount of refresh moves the CTR.
5. [content_813e88069237] score=233,561 | refresh_revie

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


In [4]:
# --- Weak picks, by number in the top 10 above ---
print("Weakest picks in the top 10:")
print("- #7 (content_b511d4bc4ad2): CTR 0.14% vs a 0.15% tier baseline -- the gap is inside")
print("  rounding noise, not a real underperformer. The rule's threshold is a hard '<', with")
print("  no minimum-gap buffer, so borderline rows like this slip in as false positives.")
print("- #6 (content_c8e9d6ab9013): a literal 0.00% CTR on 208k impressions is either a real,")
print("  serious problem or a tracking gap -- the rule can't tell those apart from this data")
print("  alone, so it should be flagged for a sanity check, not auto-actioned.")
print()

# --- Leakage / product-flag check ---
leak_terms = ["health_score", "priority_score", "action_type", "refresh_tier"]
cols_used = ["impressions_90d", "days_since_last_update", "avg_position", "ctr", "position_tier"]
found_product_flags = [c for c in leak_terms if c in df.columns]
future_window_cols = ["impressions_last_30d", "impressions_prev_30d", "trend_pct", "trend_direction"]
used_future_cols = [c for c in future_window_cols if c in cols_used]

print(f"columns the rule actually reads from: {cols_used}")
print(f"product decision flags present in the dataset: {found_product_flags} (expected: none -- not shipped)")
print(f"future-window / label-derived columns used by the rule: {used_future_cols} (expected: none)")
print()
print("trend_direction and trend_pct appear only in the top-10 review table above as context")
print("for a human reading the queue -- neither feeds the score, the gate, or the reason code.")
print("No client names, raw URLs, or raw queries appear anywhere in this notebook or the CSV.")


Weakest picks in the top 10:
- #7 (content_b511d4bc4ad2): CTR 0.14% vs a 0.15% tier baseline -- the gap is inside
  rounding noise, not a real underperformer. The rule's threshold is a hard '<', with
  no minimum-gap buffer, so borderline rows like this slip in as false positives.
- #6 (content_c8e9d6ab9013): a literal 0.00% CTR on 208k impressions is either a real,
  serious problem or a tracking gap -- the rule can't tell those apart from this data
  alone, so it should be flagged for a sanity check, not auto-actioned.

columns the rule actually reads from: ['impressions_90d', 'days_since_last_update', 'avg_position', 'ctr', 'position_tier']
product decision flags present in the dataset: [] (expected: none -- not shipped)
future-window / label-derived columns used by the rule: [] (expected: none)

trend_direction and trend_pct appear only in the top-10 review table above as context
for a human reading the queue -- neither feeds the score, the gate, or the reason code.
No client names

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.